<a href="https://colab.research.google.com/github/oleg61/DataScient/blob/Learn_AI/%D0%94%D0%97_1_%D0%92%D0%BE%D1%80%D0%BE%D0%BF%D0%B0%D0%B5%D0%B2_%D0%9E%D0%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Digit Recognizer
https://www.kaggle.com/c/digit-recognizer

In [ ]:
# Установка необходимых пакетов
!pip install -q scikit-image kaggle

# Настройка Kaggle API и загрузка данных

In [ ]:
import os
from google.colab import files

# 1. Загрузите файл kaggle.json вручную (см. инструкцию ниже)
print("📥 Пожалуйста, загрузите файл 'kaggle.json' (см. шаги ниже).")
uploaded = files.upload()  # → выберите файл kaggle.json с вашего компьютера

# 2. Переместим его в нужное место и настроим права
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# 3. Скачиваем данные конкурса Digit Recognizer
!kaggle competitions download -c digit-recognizer

# 4. Распаковываем архивы
!unzip -q digit-recognizer.zip

print("✅ Данные загружены: train.csv, test.csv")

📥 Пожалуйста, загрузите файл 'kaggle.json' (см. шаги ниже).


Saving kaggle.json to kaggle (1).json
Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 4, in <module>
    from kaggle.cli import main
  File "/usr/local/lib/python3.12/dist-packages/kaggle/__init__.py", line 6, in <module>
    api.authenticate()
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 441, in authenticate
    self._load_config(config_data)
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 492, in _load_config
    raise ValueError('Error: Missing %s in configuration.' % item)
ValueError: Error: Missing username in configuration.
✅ Данные загружены: train.csv, test.csv


## Импорты и функция HOG


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from skimage.feature import hog
from joblib import Parallel, delayed
import time

# Функция для извлечения HOG из одного изображения
def extract_hog_single(img_flat, orientations=9, pixels_per_cell=(4, 4), cells_per_block=(2, 2)):
    img_2d = img_flat.reshape(28, 28)
    return hog(
        img_2d,
        orientations=orientations,
        pixels_per_cell=pixels_per_cell,
        cells_per_block=cells_per_block,
        block_norm='L2-Hys',
        feature_vector=True
    )

# Векторизованная (параллельная) версия для всего датасета
def extract_hog_batch(images, **hog_kwargs):
    print(f"→ Извлечение HOG из {len(images)} изображений...")
    start = time.time()
    features = Parallel(n_jobs=-1)(
        delayed(extract_hog_single)(img, **hog_kwargs) for img in images
    )
    print(f"⏱️  Завершено за {time.time() - start:.1f} сек.")
    return np.array(features)

# Загрузка, предобработка и извлечение признаков

In [ ]:
# Загрузка данных
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

X_train_full = train_df.drop(columns=['label']).values.astype(np.float32)
y_train_full = train_df['label'].values
X_test = test_df.values.astype(np.float32)

print(f"Размер обучающей выборки: {X_train_full.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

# ⚙️ Параметры HOG (можно менять для экспериментов)
HOG_PARAMS = dict(
    orientations=9,
    pixels_per_cell=(4, 4),
    cells_per_block=(2, 2)
)

# Извлечение признаков
X_train_hog = extract_hog_batch(X_train_full, **HOG_PARAMS)
X_test_hog = extract_hog_batch(X_test, **HOG_PARAMS)

print(f"Размерность HOG-признаков: {X_train_hog.shape[1]}")

Размер обучающей выборки: (42000, 784)
Размер тестовой выборки: (28000, 784)
→ Извлечение HOG из 42000 изображений...
⏱️  Завершено за 50.9 сек.
→ Извлечение HOG из 28000 изображений...
⏱️  Завершено за 42.0 сек.
Размерность HOG-признаков: 1296


## Обучение и валидация

In [ ]:
# Разделение на train / val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_hog, y_train_full,
    test_size=0.15,
    random_state=42,
    stratify=y_train_full
)

# Обучение Logistic Regression
print("Обучение модели...")
model = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    C=1.0,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Оценка
y_val_pred = model.predict(X_val)
val_acc = accuracy_score(y_val, y_val_pred)
print(f"✅ Валидационная accuracy: {val_acc:.4f} (ожидаемо: 0.92–0.94)")

Обучение модели...
✅ Валидационная accuracy: 0.9727 (ожидаемо: 0.92–0.94)


## Прогноз и сохранение submission

In [ ]:
# Прогноз на тестовой выборке Kaggle
y_test_pred = model.predict(X_test_hog)

# Формирование submission
submission = pd.DataFrame({
    'ImageId': np.arange(1, len(y_test_pred) + 1),
    'Label': y_test_pred
})

submission.to_csv('submission_hog_logreg.csv', index=False)
print("✅ submission_hog_logreg.csv сохранён!")

# Показать первые 5 строк
submission.head()

✅ submission_hog_logreg.csv сохранён!


,ImageId,Label
0,1,2
1,2,0
2,3,9
3,4,0
4,5,3


# Скачать submission

In [ ]:
# Скачать файл локально
from google.colab import files
files.download('submission_hog_logreg.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>